# Gulfstream walkthrough — yield curves (`zero_rates`)

End-to-end Graph **1** and Graph **2** on real curve data, then the same pair of
pipelines again with **kernel PCA** and **DMD** embeddings.

| Part | Dimred | Pipelines |
|------|--------|-----------|
| A | PCA (default) | Graph 1 → Graph 2 (seeded from Graph 1) |
| B | Kernel PCA | Graph 1 → Graph 2 |
| C | DMD | Graph 1 → Graph 2 |

**Database:** `D:/data/duckdb/ycs_data.duckdb` · **table:** `zero_rates`

Run cells top-to-bottom. After you finish, tell the agent so outputs can be verified.


## 0. Project setup


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import duckdb
import pandas as pd
import polars as pl
from plotnine import aes, geom_line, ggplot, labs, theme_bw, facet_wrap, theme

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "pyproject.toml").exists():
    ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR.parent / "pyproject.toml").exists():
    ROOT = NOTEBOOK_DIR.parent
else:
    ROOT = Path(r"D:/Code/gulfstream")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

YCS_DB = Path(r"D:/data/duckdb/ycs_data.duckdb")
OUT_DIR = ROOT / "outputs" / "notebooks" / "ycs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT =", ROOT)
print("YCS_DB exists =", YCS_DB.exists())


## 1. Peek at `zero_rates`

Long format: one row per `(date, source)` with tenor columns `Y002p0`, `Y005p0`, …


In [ ]:
con = duckdb.connect(str(YCS_DB), read_only=True)
print("tables:", con.execute("SHOW TABLES").fetchall())
sample = con.execute(
    """
    SELECT date, source, Y002p0, Y005p0, Y010p0, Y030p0
    FROM zero_rates
    WHERE source IN ('USA', 'DEU', 'ITA')
      AND date >= '2015-01-01'
    ORDER BY date, source
    LIMIT 6
    """
).pl()
print(sample)
coverage = con.execute(
    """
    SELECT source, COUNT(*) AS n, MIN(date) AS dmin, MAX(date) AS dmax
    FROM zero_rates
    GROUP BY 1
    ORDER BY 1
    """
).pl()
print(coverage)
con.close()


## 2. Load through gulfstream

`config/sources/notebook_ycs.yaml` pivots USA/DEU/ITA tenors, joins EURUSD/GBPUSD, then runs
`generate_yield_features` (spreads, butterflies, rolling vol / corr).


In [ ]:
from gulfstream.pipelines.hamilton.driver import load_features

features_df, source_type = load_features(
    ROOT / "config" / "sources" / "notebook_ycs.yaml",
    project_root=ROOT,
)
print("source_type:", source_type)
print("shape:", features_df.shape)

from gulfstream.common import frames

print("n_features:", frames.n_features(features_df))
print("date range:", features_df["date"].min(), "→", features_df["date"].max())
print("feature sample:", frames.feature_columns(features_df)[:10])
features_df.head(3)


## 3. Explore a few series


In [ ]:
plot_cols = [
    c
    for c in [
        "USA_Y010p0",
        "DEU_Y010p0",
        "ITA_Y010p0",
        "USA_Y002p0_minus_USA_Y010p0",
        "EURUSD",
    ]
    if c in features_df.columns
]
long = (
    features_df.select(["date", *plot_cols])
    .unpivot(index="date", on=plot_cols, variable_name="series", value_name="value")
    .to_pandas()
)
long["date"] = pd.to_datetime(long["date"])

(
    ggplot(long, aes("date", "value", color="series"))
    + geom_line(size=0.4)
    + facet_wrap("~series", scales="free_y", ncol=1)
    + theme_bw()
    + theme(figure_size=(10, 2.2 * len(plot_cols)), legend_position="none")
    + labs(title="Selected yield / FX features", x="", y="")
)


## 4. Shared helpers

Reusable Graph 1 / Graph 2 runners. Graph 2 **reuses** the Graph 1 segmentation by
seeding `retrain.regimes_df` from the Graph 1 breakpoints / hierarchy.


In [ ]:
import copy
from IPython.display import Image, display

from gulfstream.common import frames, utils
from gulfstream.detection import time_index as bkpt_time
from gulfstream.metrics import regime_plots
from gulfstream.metrics.regime_plots import _visualize_market_regimes
from gulfstream.pipelines.hamilton.driver import run_segmentation_pair
from gulfstream.pipelines.graph2 import targeted_retrain_with_user_specified_df


def load_core_params(img_dir: Path) -> dict:
    """Graph 1 core YAML with notebook-friendly metrics toggles."""
    params = utils.read_config_yaml(
        str(ROOT / "config" / "graph1" / "default_core.yaml"),
        img_dir=str(img_dir),
        log_dir=str(ROOT / "outputs" / "logs"),
    )
    params["test_num"] = 0
    params["metrics"]["mode"] = "display_and_write"
    params["metrics"]["plot"] = True
    params["metrics"]["dir"] = str(img_dir)
    params["metrics"]["image_dir"] = str(img_dir)
    params["robustness"]["enabled"] = False
    params["stability"]["enabled"] = False
    return params


def with_dimred(params: dict, method: str) -> dict:
    """Return a deep copy configured for pca / kpca / dmd."""
    out = copy.deepcopy(params)
    method = method.lower()
    out["algo"]["dimred"] = [method]
    if method == "kpca":
        out["algo"]["kpca_kernel_params"] = [{"kernel": "rbf", "gamma": "median"}]
    elif method == "dmd":
        # Keep the window modest so Graph 2 regime slices still fit.
        out["algo"]["dmd_stride"] = [5]
        out["algo"]["dmd_rolling_window"] = [20]
    elif method != "pca":
        raise ValueError(f"Unsupported dimred for this notebook: {method}")
    return out


def seed_regimes_from_results(df: pl.DataFrame, res) -> pl.DataFrame:
    """Convert a Graph 1 SegmentResults into a Graph 2 ``regimes_df`` seed.

    Graph 2 reads ``End`` on all but the last row as *breakpoint dates*
    (see ``regimes_df_to_bkpts``), so ``End`` must be ``dates[bkpt]``, not the
    last observation of the preceding regime.
    """
    dates = frames.dates_series(df).to_list()
    n = len(dates)
    bkpts = sorted(int(b) for b in (res.bkpts or []) if 0 < int(b) < n)
    hierarchy = {
        int(k): int(v)
        for k, v in (res.hierarchy or {b: 1 for b in bkpts}).items()
    }
    rows = []
    for i, b in enumerate(bkpts):
        start_i = 0 if i == 0 else bkpts[i - 1]
        rows.append(
            {
                "Start": dates[start_i],
                "End": dates[b],
                "Regime": i,
                "Hierarchy Level of End": int(hierarchy.get(b, 1)),
            }
        )
    start_last = bkpts[-1] if bkpts else 0
    rows.append(
        {
            "Start": dates[start_last],
            "End": dates[n - 1],
            "Regime": len(bkpts),
            "Hierarchy Level of End": 0,
        }
    )
    return pl.DataFrame(rows)


def summarize_breakpoints(df: pl.DataFrame, res, label: str) -> None:
    dates = frames.dates_series(df).to_list()
    print(f"[{label}] kept={res.bkpts}  invalid={res.invalid_bkpts}")
    for b in res.bkpts:
        print(f"  bkpt {b} → {dates[b]}")


def show_regimes(df: pl.DataFrame, res, title: str, variables: list[str]) -> pl.DataFrame:
    dates = frames.dates_series(df).to_list()
    hierarchy = res.hierarchy or {b: 1 for b in res.bkpts}
    regimes_df = bkpt_time._get_regime_intervals(hierarchy, dates)
    vars_ = [c for c in variables if c in frames.feature_columns(df)][:2]
    _visualize_market_regimes(
        df,
        regimes_df,
        title=title,
        variables=vars_ or frames.feature_columns(df)[:2],
        valid_bkpts=res.bkpts,
        invalid_bkpts=res.invalid_bkpts,
        low_confidence_bkpts=list(res.low_confidence_bkpts or []),
        mode="display",
    )
    return regimes_df


def run_graph1(df: pl.DataFrame, params: dict, label: str, plot_vars: list[str]):
    """Hamilton single-pass (Graph 1 core). Returns (unprocessed, processed)."""
    print(f"=== Graph 1 · {label} · dimred={params['algo']['dimred']} ===")
    unproc, proc = run_segmentation_pair(df, params)
    summarize_breakpoints(df, proc, label)
    show_regimes(df, proc, f"Graph 1 · {label}", plot_vars)
    return unproc, proc


def run_graph2(
    df: pl.DataFrame,
    params: dict,
    seed_res,
    out_dir: Path,
    label: str,
    *,
    max_iter: int = 3,
    threshold: float = 1e-6,
) -> Path:
    """Graph 2 auto-retrain seeded from a Graph 1 ``SegmentResults``."""
    out_dir.mkdir(parents=True, exist_ok=True)
    g2 = copy.deepcopy(params)
    g2["metrics"]["dir"] = str(out_dir)
    g2["metrics"]["image_dir"] = str(out_dir)
    g2["metrics"]["mode"] = "display_and_write"
    g2["metrics"]["plot"] = True
    seed = seed_regimes_from_results(df, seed_res)
    print(f"=== Graph 2 · {label} · seed regimes ===")
    print(seed.to_dicts())
    g2["retrain"] = {
        "interactive": False,
        "features": ["__auto__"],
        "num_worst_features": min(5, frames.n_features(df)),
        "threshold": threshold,
        "max_iter": max_iter,
        "regimes_df": seed,
    }
    targeted_retrain_with_user_specified_df(df, g2)
    # Surface heatmaps written by the retrain loop
    pngs = sorted(out_dir.rglob("retrain_iteration_*.png"))
    if not pngs:
        pngs = sorted(out_dir.rglob("*.png"))[:6]
    print(f"Graph 2 artifacts under {out_dir} ({len(list(out_dir.rglob('*')))} files)")
    for p in pngs[:8]:
        print(" ", p.relative_to(out_dir))
        try:
            display(Image(filename=str(p)))
        except Exception as exc:
            print("  (could not display)", exc)
    return out_dir


print("Helpers ready: load_core_params, with_dimred, run_graph1, run_graph2")


---
# Part A — PCA (baseline)

Default Graph 1 core: **PCA → RFF → ruptures → MMD**, then Graph 2 targeted retrain
on the worst-L2 regime / features.


## A.1 Graph 1 (PCA)


In [ ]:
params_pca = load_core_params(OUT_DIR / "pca")
params_pca["metrics"]["features_to_plot"] = plot_cols[:3]
params_pca = with_dimred(params_pca, "pca")

unproc_pca, proc_pca = run_graph1(
    features_df, params_pca, "PCA", plot_cols
)
seed_regimes_from_results(features_df, proc_pca)


## A.2 Graph 2 (seeded from PCA Graph 1)

Auto-retrain loop: L2 heatmap → pick highest-error regime + worst features →
re-run segmentation on that slice → merge breakpoints → repeat until the loss
threshold or `max_iter`.


In [ ]:
g2_pca_dir = run_graph2(
    features_df,
    params_pca,
    proc_pca,
    OUT_DIR / "pca" / "graph2",
    "PCA",
    max_iter=3,
)


---
# Part B — Kernel PCA

Same data and test settings; only the embedding changes to **kernel PCA** (RBF).
Graph 2 again reuses *this* Graph 1 run as its seed.


## B.1 Graph 1 (kernel PCA)


In [ ]:
params_kpca = load_core_params(OUT_DIR / "kpca")
params_kpca["metrics"]["features_to_plot"] = plot_cols[:3]
# keep equity-style significance only if needed; YCS default is fine
params_kpca = with_dimred(params_kpca, "kpca")

unproc_kpca, proc_kpca = run_graph1(
    features_df, params_kpca, "kPCA", plot_cols
)


## B.2 Graph 2 (seeded from kPCA Graph 1)


In [ ]:
g2_kpca_dir = run_graph2(
    features_df,
    params_kpca,
    proc_kpca,
    OUT_DIR / "kpca" / "graph2",
    "kPCA",
    max_iter=3,
)


---
# Part C — DMD

**Dynamic Mode Decomposition** uses a rolling window + stride. The embedding is
shorter than the raw series; gulfstream handles the date alignment inside dimred.


## C.1 Graph 1 (DMD)


In [ ]:
params_dmd = load_core_params(OUT_DIR / "dmd")
params_dmd["metrics"]["features_to_plot"] = plot_cols[:3]
params_dmd = with_dimred(params_dmd, "dmd")

unproc_dmd, proc_dmd = run_graph1(
    features_df, params_dmd, "DMD", plot_cols
)


## C.2 Graph 2 (seeded from DMD Graph 1)


In [ ]:
g2_dmd_dir = run_graph2(
    features_df,
    params_dmd,
    proc_dmd,
    OUT_DIR / "dmd" / "graph2",
    "DMD",
    max_iter=3,
)


---
# Comparison

Breakpoint indices kept by each Graph 1 embedding on the same feature matrix.


In [ ]:
dates = frames.dates_series(features_df).to_list()

def bkpt_table(label, res):
    return {
        "dimred": label,
        "n_bkpts": len(res.bkpts),
        "bkpts": res.bkpts,
        "dates": [str(dates[b]) for b in res.bkpts],
        "n_invalid": len(res.invalid_bkpts),
    }

summary = pl.DataFrame(
    [
        bkpt_table("pca", proc_pca),
        bkpt_table("kpca", proc_kpca),
        bkpt_table("dmd", proc_dmd),
    ]
)
summary


## CLI equivalents

```bash
# Graph 1 PCA (default core)
uv run python -m gulfstream.cli --mode graph1 \
  --config config/graph1/default_core.yaml \
  --source-config config/sources/notebook_ycs.yaml

# Graph 2 auto-retrain (empty seed; or set retrain.regimes_df in YAML)
uv run python -m gulfstream.cli --mode graph2 \
  --config config/graph2/full_graph2.yaml \
  --source-config config/sources/notebook_ycs.yaml
```

To try kPCA / DMD from the CLI, copy `default_core.yaml` / `full_graph2.yaml` and set
`algo.dimred: [kpca]` (+ `kpca_kernel_params`) or `algo.dimred: [dmd]`
(+ `dmd_stride`, `dmd_rolling_window`).
